<a href="https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaneriah43/Flyrank_Ml_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I am using a Random Forest classifier, starting with a Decision Tree (max_depth=3) as a readable stepping stone.

My lane is Refresh / Content Opportunity Scoring — the task is ranking pages by priority, not explaining a single rule. Random Forest fits because:

My signals (impressions, CTR, position, staleness) interact in non-linear ways a single threshold cannot capture
It produces a probability score per page — exactly what a ranking needs
It handles class imbalance well with class_weight="balanced"
Feature importance gives an honest read on which signals the model actually relies on

I avoid Gradient Boosting here — my label is a proxy, not a confirmed outcome, so a simpler model that I can audit is safer than one that optimises harder against a noisy target.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
import os
import numpy as np
import getpass
import pandas as pd
import duckdb
import json
from google.colab import userdata
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

In [2]:
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): ")

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')")

In [4]:
REL = "hf://datasets/FlyRank/internship-warehouse"

In [5]:
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [14]:
df = con.execute(f"""
    WITH base AS (
        SELECT
            d.content_hash_id,
            d.client_hash_id,
            d.month,
            d.report_date,
            d.gsc_impressions,
            d.gsc_clicks,
            d.gsc_avg_position,
            MAX(d.report_date) OVER (
                PARTITION BY d.content_hash_id, d.client_hash_id
            ) AS max_date
        FROM {TABLES['fact_daily']} d
        WHERE d.month = '2026-03'
          AND d.gsc_data_available = TRUE
    )
    SELECT
        b.content_hash_id,
        b.client_hash_id,
        b.month,
        SUM(b.gsc_impressions)                                        AS impressions_90d,
        SUM(b.gsc_clicks)                                             AS clicks_90d,
        AVG(b.gsc_avg_position)                                       AS avg_position,
        SUM(b.gsc_clicks) / NULLIF(SUM(b.gsc_impressions), 0)        AS ctr,
        SUM(CASE WHEN b.report_date >= (b.max_date - INTERVAL 30 DAYS)
                 THEN b.gsc_impressions ELSE 0 END)                   AS impressions_last30,
        SUM(CASE WHEN b.report_date < (b.max_date - INTERVAL 30 DAYS)
                 THEN b.gsc_impressions ELSE 0 END)                   AS impressions_prev30,
        c.content_type,
        c.word_count,
        -- FIXED: max_date minus last_optimized_date (not the other way)
        DATEDIFF('day', c.last_optimized_date, MAX(b.report_date))   AS days_since_last_update,
        DATEDIFF('day', c.content_created_date, MAX(b.report_date))  AS content_age_days
    FROM base b
    JOIN {TABLES['dim_content']} c
        ON b.content_hash_id = c.content_hash_id
    GROUP BY
        b.content_hash_id, b.client_hash_id, b.month,
        c.content_type, c.word_count,
        c.last_optimized_date, c.content_created_date
""").df()

# Fix negative values — take absolute value as safety net
df["days_since_last_update"] = df["days_since_last_update"].abs()
df["content_age_days"]       = df["content_age_days"].abs()

# Derive trend_direction
df["trend_direction"] = np.where(
    df["impressions_last30"] > df["impressions_prev30"], "up",
    np.where(df["impressions_last30"] < df["impressions_prev30"], "down", "flat")
)

# Check distributions before setting label
print("days_since_last_update stats:")
print(df["days_since_last_update"].describe())
print(f"\nValues >= 180: {(df['days_since_last_update'] >= 180).sum():,}")
print(f"Values >= 90:  {(df['days_since_last_update'] >= 90).sum():,}")
print(f"Values >= 30:  {(df['days_since_last_update'] >= 30).sum():,}")

print("\ntrend_direction distribution:")
print(df["trend_direction"].value_counts())

print("\nImpressions median:", df["impressions_90d"].median())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

days_since_last_update stats:
count      39764.0
mean     70.476008
std        16.1373
min           24.0
25%           56.0
50%           72.0
75%           83.0
max          125.0
Name: days_since_last_update, dtype: Float64

Values >= 180: 0
Values >= 90:  6,079
Values >= 30:  39,328

trend_direction distribution:
trend_direction
up    176738
Name: count, dtype: int64

Impressions median: 173.0


In [16]:
# Derive trend from ctr vs position — since last30/prev30 split doesn't work for this month
df["ctr_opportunity"] = np.where(
    df["avg_position"] <= 10,
    df["ctr"] < df.groupby(
        pd.cut(df["avg_position"], bins=[0, 3, 5, 10, 20, 100])
    )["ctr"].transform("median"),
    False
)

# Label — visible page with below-median CTR at its position tier
df["is_declining_label"] = (
    (df["impressions_90d"] >= df["impressions_90d"].median()) &
    (df["ctr_opportunity"] == True)
).astype(int)

print(f"Positive labels: {df['is_declining_label'].sum():,} ({df['is_declining_label'].mean()*100:.1f}%)")
print(f"Negative labels: {(df['is_declining_label']==0).sum():,}")
print(f"Columns: {df.columns.tolist()}")

Positive labels: 3,416 (1.9%)
Negative labels: 173,322
Columns: ['content_hash_id', 'client_hash_id', 'month', 'impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'impressions_last30', 'impressions_prev30', 'content_type', 'word_count', 'days_since_last_update', 'content_age_days', 'trend_direction', 'ctr_opportunity', 'is_declining_label']


/tmp/ipykernel_2173/4216250255.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["ctr"] < df.groupby(


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a grouped split by client — all pages belonging to one client go entirely into train or entirely into test. This is the honest choice for this lane because:

* Each client has a distinct content strategy and traffic profile
* A random split would let the model learn client-specific patterns and score the same client's pages at test time — inflating Precision@50
* The real-world use case is scoring a new client's pages — the model must generalise across clients, not within them

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [22]:
# Reassign y from the now-correct label column
y = df["is_declining_label"].values

print(f"Positive labels: {y.sum():,} ({y.mean()*100:.1f}%)")
print(f"Negative labels: {(y==0).sum():,}")

# Re-run stratified split
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"\nTrain rows:    {len(train_idx):,}")
print(f"Test rows:     {len(test_idx):,}")
print(f"Positive rate — train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")
print(f"Positives in train: {y_train.sum():,}")
print(f"Positives in test:  {y_test.sum():,}")

Positive labels: 3,416 (1.9%)
Negative labels: 173,322

Train rows:    141,390
Test rows:     35,348
Positive rate — train: 0.019 | test: 0.019
Positives in train: 2,733
Positives in test:  683


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [23]:
# --- Baseline (Week 4 hand rule) — updated to match new label ---
stale           = (df["days_since_last_update"].fillna(0) >= 90).astype(int).values
visible         = (df["impressions_90d"].fillna(0) >= df["impressions_90d"].median()).astype(int).values
underperforming = (df["ctr_opportunity"] == True).astype(int).values
hand_score      = stale * visible * underperforming * df["impressions_90d"].fillna(0).values
hand_test       = hand_score[test_idx]

print(f"Hand rule — pages flagged: {(hand_score > 0).sum():,}")

# --- Decision Tree ---
tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)

# Guard against single-class training
if tree.n_classes_ < 2:
    print("ERROR: only one class in training data — check label definition")
else:
    tree_test = tree.predict_proba(X_test)[:, 1]

    # --- Random Forest ---
    rf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                 max_depth=5, random_state=42)
    rf.fit(X_train, y_train)
    rf_test = rf.predict_proba(X_test)[:, 1]

    # --- Results table ---
    print(f"\n{'Method':<20} {'P@20':>6} {'P@50':>6}")
    print("-" * 35)
    for name, scores in [("Hand rule",     hand_test),
                          ("Decision Tree", tree_test),
                          ("Random Forest", rf_test)]:
        p20 = precision_at_k(scores, y_test, 20)
        p50 = precision_at_k(scores, y_test, 50)
        print(f"{name:<20} {p20:>6.3f} {p50:>6.3f}")

    # Save metrics
    os.makedirs("work/outputs", exist_ok=True)
    metrics = {
        "hand_rule":     {"p@20": precision_at_k(hand_test, y_test, 20),
                          "p@50": precision_at_k(hand_test, y_test, 50)},
        "decision_tree": {"p@20": precision_at_k(tree_test, y_test, 20),
                          "p@50": precision_at_k(tree_test, y_test, 50)},
        "random_forest": {"p@20": precision_at_k(rf_test, y_test, 20),
                          "p@50": precision_at_k(rf_test, y_test, 50)},
    }
    with open("work/outputs/metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print("\nMetrics saved → work/outputs/metrics.json")

Hand rule — pages flagged: 154

Method                 P@20   P@50
-----------------------------------
Hand rule             1.000  0.640
Decision Tree         0.500  0.640
Random Forest         1.000  1.000

Metrics saved → work/outputs/metrics.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [24]:
# Feature importance
importances = pd.Series(rf.feature_importances_, index=features)\
                .sort_values(ascending=False)
print("=== Feature Importance ===")
print(importances.round(4))

# Decision tree rules — readable logic
print("\n=== Decision Tree Rules (depth=3) ===")
print(export_text(tree, feature_names=features))

# False negatives — missed positives outside top 50
rf_ranking   = np.argsort(-rf_test)
missed_idx   = rf_ranking[50:]
missed_pos   = missed_idx[y_test[missed_idx] == 1]

print(f"\nFalse negatives (positives missed outside top 50): {len(missed_pos):,}")

if len(missed_pos) > 0:
    missed_df = pd.DataFrame(X_test[missed_pos], columns=features)
    print(missed_df.describe().round(2))

# False positives — wrong calls in top 50
top50_idx    = rf_ranking[:50]
fp_idx       = top50_idx[y_test[top50_idx] == 0]
print(f"\nFalse positives (wrong calls in top 50): {len(fp_idx):,}")

if len(fp_idx) > 0:
    fp_df = pd.DataFrame(X_test[fp_idx], columns=features)
    print(fp_df.describe().round(2))

=== Feature Importance ===
avg_position              0.5081
impressions_90d           0.2602
ctr                       0.1803
clicks_90d                0.0344
days_since_last_update    0.0090
word_count                0.0041
content_age_days          0.0038
dtype: float64

=== Decision Tree Rules (depth=3) ===
|--- avg_position <= 5.00
|   |--- ctr <= 0.00
|   |   |--- impressions_90d <= 172.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  172.50
|   |   |   |--- class: 1
|   |--- ctr >  0.00
|   |   |--- days_since_last_update <= 14.50
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  14.50
|   |   |   |--- class: 0
|--- avg_position >  5.00
|   |--- class: 0


False negatives (positives missed outside top 50): 633
        impressions_90d  clicks_90d  avg_position    ctr  \
count             633.0       633.0        633.00  633.0   
unique            520.0        23.0        633.00  172.0   
top               426.0         0.0          4.09    0.0   
freq   

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.